[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/kitaev_wire/kitaev_wire.ipynb)

# Moving a Majorana boundary mode

A superconducting wire can support a mode at each end of a topological segment. Their separation suppresses the energy splitting of a shared fermionic degree of freedom. Gates move those ends. A sufficiently slow sweep carries a mode along; a fast sweep loses it to the bulk. We build the wire's couplings, solve its boundary modes, and evolve one of them while the gate moves.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor
from examples.quantum.kitaev_wire import render
from examples.quantum.kitaev_wire.core import smooth_progress

In [ ]:
ga = Algebra("x+y+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Majorana = ga.gatype.vector()
# x and y label the two Majorana components at each site, not spatial directions.
Map = ga.gatype((Majorana, Majorana))

Majorana generators square to one and anticommute: they form a Clifford basis. A quadratic fermionic evolution is generated by a bivector, whose commutator acts as a skew map on these directions. Here each sparse cell maps two real components at an input site to two at an output site. The field map assembles their action over the wire.

In [ ]:
site_count = 96
sites = np.arange(site_count)
site_index = mv.scalar(sites[:, None]).field()           # Scalar[sites]
hopping = 1.0
pairing = 0.8
outside = 3.5
inside = 1.4
width = 1.2
left = 18.0
right = 82.0
spectral_samples = 100
energy_count = 8


def gate(sites: np.ndarray, left: np.ndarray, right: np.ndarray,
         inside: np.ndarray, outside: float, width: float) -> Scalar:
    # A smooth well in chemical potential, with the exterior held fixed.
    window = (np.tanh((sites - left[..., None]) / width)
              - np.tanh((sites - right[..., None]) / width)) / 2
    return mv.scalar((outside + (inside[..., None] - outside) * window)[..., None]).field()

# Hopping and p-wave pairing add in one cross-site coupling and subtract in the other.
# A directed bond reads a component on the next site and returns one on this site.
bond = ((hopping + pairing) * mv.y * (mv.x | Majorana)
        + (pairing - hopping) * mv.x * (mv.y | Majorana))
forward = SparseExtensor(bond.broadcast_to((site_count - 1,)),
                         sites[:-1], sites[1:], (site_count, site_count))
bonds = forward - forward.adjoint()                       # skew field map
unit = SparseExtensor.from_diagonal(mv.scalar(np.ones((site_count, 1))).field())
identity = unit * Majorana


def generator(potential: Scalar) -> SparseExtensor:
    # Chemical potential turns the two components on the same site.
    local = -potential * mv.xy.commutator(Majorana)
    return bonds + SparseExtensor.from_diagonal(local)


def modes(potential: Scalar, count: int) -> tuple[Scalar, Majorana]:
    turning = generator(potential)
    # Squaring the norm of the turning map gives squared excitation energies.
    _, states = turning.adjoint()(turning).eigh(identity, count)
    action = SparseExtensor(turning.cells[..., None, :], turning.rows,
                            turning.columns, turning.shape)
    energies = action(states).scalar_norm_squared().batch().sum(axis=-1).square_root()
    return energies, states


chemical_potentials = np.linspace(outside, inside, spectral_samples)
potential = gate(sites, np.array(left), np.array(right), chemical_potentials, outside, width)
energies, states = modes(potential, energy_count)          # [samples, modes] Scalar, [samples, modes] Majorana[sites]
# Each excitation has two real quadratures. Their combined weight is basis-independent.
density = states[:, :2].scalar_norm_squared().mean(axis=-1)
render.draw_formation(chemical_potentials, energies, density, hopping);

Lowering the chemical potential inside the gate creates a topological segment. For a uniform Kitaev chain with nonzero pairing, the transition is at $|\mu|=2t$; the dotted line marks that bulk threshold. Two low-energy modes separate from the other excitations and concentrate at the segment's boundaries. To resolve left and right, embed their two-dimensional space into the wire and pull the site-index observable back through the embedder's adjoint.

In [ ]:
destination = 48.0
durations = np.array([180.0, 12.0])
names = ("slow transport", "fast transport")
steps = 3600
stride = 20
duration_ms = 55
progress = np.arange(steps // stride + 1) * stride / steps
left_boundary = left + (destination - left) * smooth_progress(progress)
potentials = gate(sites, left_boundary, np.array(right), np.array(inside), outside, width)
_, boundary_modes = modes(potentials, 2)                  # [frames, modes] Majorana[sites]


def localize(states: Majorana) -> Majorana:
    coordinates = stack([mv.x, mv.y])
    embedder = (states * (coordinates | Majorana)).sum(axis=-1)
    # The eigenvectors of this small map select the left and right combinations.
    position = embedder.adjoint()(site_index * embedder)
    _, directions = position.eigh()
    return embedder[..., None](directions)


localized = localize(boundary_modes)                     # [frames, ends] Majorana[sites]
midpoint_progress = (np.arange(steps) + 0.5) / steps
midpoint_left = left + (destination - left) * smooth_progress(midpoint_progress)
midpoint_potentials = gate(sites, midpoint_left, np.array(right), np.array(inside), outside, width)


def evolution(initial: Majorana) -> Iterator[Majorana]:
    state = initial.broadcast_to((len(durations),))
    dt = durations / steps
    yield state
    for index, potential in enumerate(midpoint_potentials):
        turning = generator(potential)
        half_step = SparseExtensor(turning.cells * (dt[:, None] / 2),
                                   turning.rows, turning.columns, turning.shape)
        # A Cayley step of a skew map is orthogonal: it preserves the mode norm.
        # Evaluating the gate halfway through the step gives second-order evolution.
        state = (identity - half_step).solve(state + half_step(state))
        if (index + 1) % stride == 0:
            yield state


history = stack(evolution(localized[0, 0]))              # [frames, speeds] Majorana[sites]
# Project the evolving field onto both instantaneous boundary modes.
overlap = history[:, :, None].scalar_product(boundary_modes[:, None]).batch().sum(axis=-1)
share = overlap.squared().sum(axis=-1)                    # [frames, speeds] Scalar
render.inline(render.animate_transport(progress, potentials, history, localized[:, 0],
                                        share, durations, names, hopping), duration_ms)

Both sweeps follow the same gate path, in different physical times. The dashed profile is the instantaneous localized mode; the solid profile is actually evolved. The pale region marks where the local chemical potential lies in the topological regime. The slow sweep follows the boundary, while the fast sweep releases a spreading field. Bulk weight is the share outside the instantaneous two-dimensional boundary-mode space.

In [ ]:
render.draw_transport(progress, history, potentials, durations, names, hopping);

Bring the boundaries together and their modes overlap. The left and right combinations then couple, lifting the lowest excitation away from zero. The curve below is a sequence of stationary eigenproblems, not another time evolution. Once the overlap is appreciable, the localized combinations are not individually energy eigenstates.

In [ ]:
separations = np.linspace(4.0, 32.0, 100)
profile_separations = np.array([24.0, 12.0, 5.0])

closing_potentials = gate(sites, right - separations, np.array(right),
                          np.array(inside), outside, width)
closing_energies, _ = modes(closing_potentials, 2)
splitting = closing_energies.mean(axis=-1)
profile_potentials = gate(sites, right - profile_separations, np.array(right),
                          np.array(inside), outside, width)
_, profile_modes = modes(profile_potentials, 2)
render.draw_overlap(separations, profile_separations, splitting, profile_potentials,
                     localize(profile_modes), hopping);

The plotted weight is squared Majorana-mode amplitude, not charge density or a quasiparticle population. This is a clean spinless Kitaev chain with prescribed real p-wave pairing and externally controlled chemical potential; it does not solve for the superconducting order parameter or include interactions and dissipation. [Kitaev's paper](https://arxiv.org/abs/cond-mat/0010440) gives the chain and its boundary modes. [Alicea and collaborators](https://arxiv.org/abs/1006.4395) develop gate-controlled transport and exchange through wire networks.